# Machine Translation

In [17]:
from datasets import load_dataset
from transformers import MarianMTModel, MarianTokenizer, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import pandas as pd
import torch
import os
import transformers


### Loading the Dutch–English Dataset

We use the OPUS100 dataset from Hugging Face Datasets.  
Although it only provides the en-nl configuration, each entry contains both English (en) and Dutch (nl) sentences.  
By treating Dutch (nl) as the source and English (en) as the target, we can effectively train a Dutch -> English translation model.

In [18]:
dataset = load_dataset("opus100", "en-nl")

print(dataset)
print(dataset["train"][0])


DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})
{'translation': {'en': "It's for you.", 'nl': 'Het is voor jou.'}}


### Choosing the Translation Model

We use the MarianMT model from the `Helsinki-NLP` group (`opus-mt-nl-en`).  

- MarianMT is a Transformer-based sequence-to-sequence model designed for machine translation.  
- It has already been pretrained on large Dutch <--> English set.  
- Fine-tuning it on OPUS100 adapts the model to our domain (e.g., transcripts, specialized text).  

This saves compute resources compared to training a translation model from scratch.

In [19]:
model_name = "Helsinki-NLP/opus-mt-nl-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)


## Preprocessing

The dataset provides samples in the form:

```python
{"translation": {"en": "...", "nl": "..."}}
```
We need to prepare the dataset for training:

- Inputs (source language): take the Dutch sentences (`nl`).
- Targets (target language): take the English sentences (`en`).

**Tokenization:**
  - The tokenizer converts raw text into input IDs the model can understand.
  - We tokenize Dutch sentences as model inputs.
  - We tokenize English sentences as targets using `as_target_tokenizer`, so the model knows these are labels.

**Padding and truncation:**  
  All sequences are paddedto a fixed length (128 tokens) to keep batches consistent.

In [ ]:
max_length = 128

def preprocess_function(examples):
    inputs = [ex["nl"] for ex in examples["translation"]]
    targets = [ex["en"] for ex in examples["translation"]]
    model_inputs = tokenizer(inputs, max_length=max_length, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_length, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)


### Training Setup

We use the Hugging Face Seq2SeqTrainer for fine-tuning.  

Key hyperparameters:
- `learning_rate = 5e-5`: a common starting point for fine-tuning Transformers.  
- `batch_size = 16`: balances speed and GPU memory usage.  
- `num_train_epochs = 1`: one epoch for quick experimentation (can increase for better results).  
- `weight_decay = 0.01`: helps prevent overfitting.  
- `evaluation_strategy = "epoch"`: evaluate once per epoch to monitor performance.  

We also set `predict_with_generate=True` so the trainer uses the model’s `generate()` method during evaluation, producing actual translations.

In [21]:
batch_size = 16

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1,
    logging_dir="./logs",
    logging_steps=50,
    predict_with_generate=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(5000)),
    eval_dataset=tokenized_datasets["validation"].select(range(500)),
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model("./mt")
tokenizer.save_pretrained("./mt")


C:\Users\19102\AppData\Local\Temp\ipykernel_18508\889296543.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
c:\Users\19102\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.387400
100,0.133000
150,0.129700
200,0.128800
250,0.125800
300,0.134500


c:\Users\19102\anaconda3\Lib\site-packages\transformers\modeling_utils.py:4034: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[67027]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  # If the model config has set attributes that should be in the generation config, move them there.


('./mt\\tokenizer_config.json',
 './mt\\special_tokens_map.json',
 './mt\\vocab.json',
 './mt\\source.spm',
 './mt\\target.spm',
 './mt\\added_tokens.json')

In [ ]:
src_path = "../Task 7/Translations/Source.nl"
with open(src_path, "r", encoding="utf-8") as f:
    src_lines = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(src_lines)} sentences from transcript.")


Loaded 299879 sentences from transcript.


### Translating New Sentences

After fine-tuning, we load the saved model and run translation:

- Batch the input sentences for efficiency.  
- Tokenize and move inputs to GPU.  
- Generate translations with `model.generate()`.  
- Decode token IDs back into text.  

In [24]:
import torch

# Pick device (GPU if available, else CPU fallback)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load fine-tuned model + tokenizer
finetuned_model = MarianMTModel.from_pretrained("./mt").to(device)
finetuned_tokenizer = MarianTokenizer.from_pretrained("./mt")

translations = []
batch_size = 16

for i in range(0, len(src_lines), batch_size):
    batch = src_lines[i:i+batch_size]
    
    # Tokenize and move input tensors to GPU
    inputs = finetuned_tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
    
    with torch.no_grad():
        translated = finetuned_model.generate(**inputs)
    
    # Decode stays on CPU (safe)
    decoded = [finetuned_tokenizer.decode(t, skip_special_tokens=True) for t in translated]
    translations.extend(decoded)

# Save to CSV
df = pd.DataFrame({"Source": src_lines, "Translation": translations})
out_path = "/Translations/translations_output.csv"
df.to_csv(out_path, index=False, encoding="utf-8")

print(f"Translations saved to {out_path}")
print(df.head(10))

KeyboardInterrupt: 